In [8]:
import pandas as pd

In [9]:
df=pd.read_csv("/content/ball_by_ball_ipl_data.csv")

In [10]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [11]:
df = df[~df['batsman_run'].isin([3, 5])]

# Convert to better target (3-class problem)
def categorize_runs(x):
    if x == 0:
        return 0   # dot ball
    elif x in [1, 2]:
        return 1   # running
    else:
        return 2   # boundary (4,6)

df['run_class'] = df['batsman_run'].apply(categorize_runs)

In [12]:
df = df[[
    'batter_name',
    'bowler_name',
    'non_striker_name',
    'batting_team',
    'bowling_team',
    'run_class'
]]

In [13]:
le = LabelEncoder()

for col in ['batter_name', 'bowler_name', 'non_striker_name', 'batting_team', 'bowling_team']:
    df[col] = le.fit_transform(df[col])

In [14]:
X = df.drop('run_class', axis=1)
y = df['run_class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [15]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features='sqrt',
    class_weight='balanced_subsample',
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

print("=== Random Forest ===")
print("Accuracy:", accuracy_score(y_test, rf_pred))
print(classification_report(y_test, rf_pred))

=== Random Forest ===
Accuracy: 0.40558179785814735
              precision    recall  f1-score   support

           0       0.45      0.40      0.42     22004
           1       0.51      0.44      0.47     24132
           2       0.20      0.33      0.25      9330

    accuracy                           0.41     55466
   macro avg       0.39      0.39      0.38     55466
weighted avg       0.43      0.41      0.42     55466



In [17]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)

xgb_pred = xgb_model.predict(X_test)

print("ADABoost")
print("Accuracy:", accuracy_score(y_test, xgb_pred))
print(classification_report(y_test, xgb_pred))

ADABoost
Accuracy: 0.4769949158042765
              precision    recall  f1-score   support

           0       0.46      0.45      0.46     22004
           1       0.49      0.68      0.57     24132
           2       0.30      0.00      0.01      9330

    accuracy                           0.48     55466
   macro avg       0.42      0.38      0.34     55466
weighted avg       0.45      0.48      0.43     55466

